# GPU jobs in the JupyDo job queue

GPUs are **only available to jobs**, not to your interactive environment: ask for them with `--gpus N`.
The job sees only the GPUs it was given, and they are free for others as soon as it ends.

How a job works (see also `jobqueue_python.ipynb`):
- it runs in a **new container**, made from a snapshot of this environment taken when you submit it:
  same software, **also what you installed yourself** (e.g. PyTorch with `pip install --user`);
- variables of this notebook are **not** available in the job: save results to files in `/sharedFolder`.

If the server has no GPUs, a job asking for `--gpus` is refused.

## 1. Is there a GPU for me?
`%jobs` shows the queue resources, GPUs included.

In [ ]:
%jobs

## 2. First GPU job: which GPU did I get?
The job prints the GPU it was given and the output of `nvidia-smi`.

In [ ]:
%%job --time 5m --cpus 1 --mem 2 --gpus 1 --name gpu_check
import os, shutil, subprocess
print('GPU given to this job:', os.environ.get('NVIDIA_VISIBLE_DEVICES'))
if shutil.which('nvidia-smi') is None:
    raise SystemExit('nvidia-smi not found: this job got no GPU (no --gpus, or no NVIDIA GPU on the server)')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

Follow it with `%jobs`, then read its output with `%job_log <id>` (replace `<id>` with the id printed above).

In [ ]:
%jobs

In [ ]:
# %job_log <id>

## 3. A real computation on the GPU (PyTorch)

PyTorch is not part of every image: install it once **in your environment** (it goes in your home, `~/.local`).
The next jobs you submit include it, because they start from a snapshot of your environment.

The command below installs the version for recent NVIDIA GPUs (CUDA 12.8). It downloads a few GB: run it once.

In [ ]:
# run once, then restart the kernel (Kernel > Restart Kernel)
# %pip install --user torch --index-url https://download.pytorch.org/whl/cu128

The job multiplies two large matrices on the GPU and on the CPU, saves the timings to a file and prints them.

In [ ]:
%%job --time 15m --cpus 4 --mem 16 --gpus 1 --name gpu_torch
import json, os, time
try:
    import torch
except ImportError:
    raise SystemExit('PyTorch is not installed in this environment: run the install cell above, '
                     'then submit this job again.')
if not torch.cuda.is_available():
    raise SystemExit('No usable GPU in this job (did you ask for --gpus?)')

def timed_matmul(device, n=8000):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    a @ b
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return time.time() - start

gpu_name = torch.cuda.get_device_name(0)
timed_matmul(torch.device('cuda'), 1000)  # warm-up
results = {'gpu': gpu_name,
           'gpu_seconds': timed_matmul(torch.device('cuda')),
           'cpu_seconds': timed_matmul(torch.device('cpu'))}
os.makedirs('results', exist_ok=True)
with open('results/gpu_benchmark.json', 'w') as f:
    json.dump(results, f, indent=1)
print(results)

In [ ]:
# when %jobs says 'done':
import json
with open('results/gpu_benchmark.json') as f:
    r = json.load(f)
print(f"{r['gpu']}: GPU {r['gpu_seconds']:.3f} s, CPU {r['cpu_seconds']:.3f} s "
      f"({r['cpu_seconds'] / r['gpu_seconds']:.0f}x faster on the GPU)")

## 4. Cancel a job
`%job_cancel <id>` cancels a pending job or stops a running one: its GPU is freed at once.

In [ ]:
# %job_cancel <id>